# ML-10 — Content Action Playbook

This notebook translates the Week 5 model into a ranked action queue the content team can actually use. Every claim here matches the evidence — no causal language, no private data.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score,
)

RAW = Path("../../data/raw/content_refresh_anonymized.csv")
OUTPUT_DIR = Path("../../work/outputs")
FIG_DIR = Path("../../work/figures")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
RANDOM_STATE = 42

df_raw = pd.read_csv(RAW)
print(f"Loaded {len(df_raw):,} rows x {df_raw.shape[1]} columns")
print(f"Clients: {df_raw['client_id'].nunique()}")

Loaded 30,000 rows x 44 columns
Clients: 32


In [2]:
# --- Prepare features (same pipeline as Week 5) ---
df = df_raw.copy()
df['is_declining_label'] = df['trend_direction'].str.lower().eq('down').astype(int)

df['log_impressions_90d'] = np.log1p(df['impressions_90d'])
df['log_clicks_90d'] = np.log1p(df['clicks_90d'])
df['log_sessions_90d'] = np.log1p(df['sessions_90d'])
df['log_ai_sessions_90d'] = np.log1p(df['ai_sessions_90d'])
df['has_clicks'] = (df['clicks_90d'] > 0).astype(int)
df['has_ai_sessions'] = (df['ai_sessions_90d'] > 0).astype(int)
df['measurable_opportunity'] = ((df['impressions_90d'] >= 100) & (df['sessions_90d'] > 0)).astype(int)

df = df[(df['impressions_90d'] > 0) & (df['content_age_days'] >= 90)].copy()
df = df.drop_duplicates(subset=['content_id']).reset_index(drop=True)

print(f"After filtering: {len(df):,} rows")
print(f"Declining rate: {df['is_declining_label'].mean():.1%}")

After filtering: 30,000 rows
Declining rate: 54.2%


In [3]:
NUMERIC_FEATURES = [
    'search_volume', 'competition', 'cpc',
    'word_count', 'char_count',
    'log_impressions_90d', 'log_clicks_90d', 'log_sessions_90d', 'log_ai_sessions_90d',
    'days_with_impressions', 'days_with_sessions',
    'content_age_days', 'days_since_last_update',
    'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct',
]

CATEGORICAL_FEATURES = [
    'competition_level', 'content_type', 'main_intent',
    'age_tier', 'freshness_tier', 'word_count_tier',
    'impression_tier', 'position_tier',
]

for col in NUMERIC_FEATURES:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').replace([np.inf, -np.inf], np.nan).fillna(0)

for col in CATEGORICAL_FEATURES:
    if col in df.columns:
        df[col] = df[col].fillna('unknown').astype(str).replace({'': 'unknown', 'nan': 'unknown'})

print(f"Numeric features: {len([c for c in NUMERIC_FEATURES if c in df.columns])}")
print(f"Categorical features: {len([c for c in CATEGORICAL_FEATURES if c in df.columns])}")

Numeric features: 18
Categorical features: 8


In [4]:
def build_X(frame):
    num_cols = [c for c in NUMERIC_FEATURES if c in frame.columns]
    cat_cols = [c for c in CATEGORICAL_FEATURES if c in frame.columns]
    X_num = frame[num_cols].apply(pd.to_numeric, errors='coerce').replace([np.inf, -np.inf], np.nan).fillna(0)
    X_cat = frame[cat_cols].fillna('unknown').astype(str)
    X_cat_enc = pd.get_dummies(X_cat, prefix=cat_cols, dummy_na=False, dtype=float)
    return pd.concat([X_num.reset_index(drop=True), X_cat_enc.reset_index(drop=True)], axis=1)

X_full = build_X(df)
ALL_COLS = sorted(X_full.columns)
X_full = X_full.reindex(columns=ALL_COLS, fill_value=0)

y_full = df['is_declining_label'].values
print(f"Feature matrix: {X_full.shape[1]} features, {X_full.shape[0]:,} rows")

Feature matrix: 52 features, 30,000 rows


In [5]:
# --- Train Logistic Regression on all data (for scoring) ---
lr_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(
        class_weight='balanced',
        max_iter=1000,
        random_state=RANDOM_STATE,
    )),
])
lr_pipeline.fit(X_full, y_full)
model_probs = lr_pipeline.predict_proba(X_full)[:, 1]

print(f"Logistic Regression trained on {X_full.shape[0]:,} rows")
print(f"Base rate (declining): {y_full.mean():.1%}")
print(f"ROC-AUC (in-sample): {roc_auc_score(y_full, model_probs):.3f}")

Logistic Regression trained on 30,000 rows


Base rate (declining): 54.2%
ROC-AUC (in-sample): 0.720


In [6]:
# --- Build the ranked action queue ---
queue = df[['content_id', 'client_id', 'content_type',
            'impressions_90d', 'avg_position', 'ctr',
            'days_since_last_update', 'content_age_days',
            'is_declining_label']].copy()
queue['model_score'] = model_probs
queue = queue.sort_values('model_score', ascending=False).reset_index(drop=True)
queue['rank'] = range(1, len(queue) + 1)

# Assign reason codes based on score and content characteristics
def assign_reason(row):
    if row['model_score'] >= 0.65:
        return "high_decline_risk"
    elif row['model_score'] >= 0.55:
        return "moderate_decline_risk"
    elif row['days_since_last_update'] >= 180 and row['impressions_90d'] >= 500:
        return "stale_visible"
    elif row['avg_position'] > 0 and row['avg_position'] <= 10 and row['ctr'] < 0.5:
        return "low_ctr_top10"
    else:
        return "monitor"

def assign_action(reason):
    action_map = {
        "high_decline_risk": "refresh_priority",
        "moderate_decline_risk": "refresh",
        "stale_visible": "refresh_stale",
        "low_ctr_top10": "refresh_ctr_fix",
        "monitor": "monitor",
    }
    return action_map.get(reason, "monitor")

queue['reason_code'] = queue.apply(assign_reason, axis=1)
queue['action'] = queue['reason_code'].apply(assign_action)

print(f"Queue length: {len(queue):,}")
print(f"\nAction distribution:")
print(queue['action'].value_counts().to_string())

Queue length: 30,000

Action distribution:
action
monitor             11400
refresh_priority     7887
refresh              5646
refresh_ctr_fix      5063
refresh_stale           4


In [7]:
# --- Show the top 30 items the team should look at first ---
top30 = queue.head(30)[['rank', 'content_id', 'action', 'reason_code',
                        'model_score', 'impressions_90d', 'avg_position', 'ctr',
                        'days_since_last_update', 'content_type']].copy()
top30['model_score'] = top30['model_score'].round(3)
top30['ctr'] = top30['ctr'].round(2)
top30['avg_position'] = top30['avg_position'].round(1)
print("=== Top 30 items to review first ===")
print("(ranked by model decline probability)")
top30

=== Top 30 items to review first ===
(ranked by model decline probability)


,rank,content_id,action,reason_code,model_score,impressions_90d,avg_position,ctr,days_since_last_update,content_type
0,1,content_f986bd514b6e,refresh_priority,high_decline_risk,0.947,22456,6.6,0.00,20,keyword article
1,2,content_7c2869a87415,refresh_priority,high_decline_risk,0.939,5271,15.2,0.00,20,keyword article
2,3,content_c8e9d6ab9013,refresh_priority,high_decline_risk,0.936,208678,9.7,0.00,104,keyword article
3,4,content_c89e3b5466ba,refresh_priority,high_decline_risk,0.931,2321,11.8,0.00,20,keyword article
4,5,content_c82bc0c24241,refresh_priority,high_decline_risk,0.929,13676,4.3,0.00,8,keyword article
5,6,content_ae6d1339904d,refresh_priority,high_decline_risk,0.929,17622,19.5,0.00,20,keyword article
6,7,content_6844e78ecb37,refresh_priority,high_decline_risk,0.924,248,5.4,0.00,104,keyword article
7,8,content_ba60decb51d4,refresh_priority,high_decline_risk,0.924,252,6.1,0.00,104,keyword article
8,9,content_caa0b9ca6768,refresh_priority,high_decline_risk,0.921,2227,9.9,0.00,20,keyword article
9,10,content_5a1e444ef843,refresh_priority,high_decline_risk,0.920,245,12.8,0.00,20,keyword article


**How to read this queue:** Each row shows a content item, its estimated decline probability from the logistic regression model, and a reason code. The reason code captures the primary signal the model relied on. "high\_decline\_risk" means the model assigned ≥ 65% decline probability — these are the items where a content refresh is most directionally supported by the data. "monitor" items scored below 55% — no action needed now, but they stay in the queue for the next scoring cycle.

### Archetype → Action mapping

The queue assigns each item a **reason code** and an **action**. The table below maps each archetype to its recommended action and explains why the action is appropriate. These are **human-reviewed recommendations**, not automatic triggers.

| Archetype | Reason code | Recommended action | Why this action | Evidence strength |
|---|---|---|---|---|
| **High decline risk** | `high_decline_risk` (score ≥ 0.65) | **refresh\_priority** — review first | The model observed multiple signals (low CTR, high impressions, stale content) strongly associated with decline in this dataset. A human should verify the page is still live and business-relevant before refreshing. | Strong — multiple features align |
| **Moderate decline risk** | `moderate_decline_risk` (score 0.55–0.65) | **refresh** — review next | The model observed a moderate association with decline. Worth reviewing after high-priority items, but the signal is less certain. | Moderate — single threshold |
| **Stale + visible** | `stale_visible` (≥ 180 days since update, ≥ 500 impressions) | **refresh\_stale** — freshness review | Content has not been updated in 6+ months and still receives meaningful impressions. The Week 4 baseline flagged this pattern. A human should assess whether the content is outdated. | Directional — rule-based signal |
| **Low CTR in top 10** | `low_ctr_top10` (avg position ≤ 10, CTR < 0.5%) | **refresh\_ctr\_fix** — CTR review | The page ranks well but underperforms on click-through. This may indicate a title/meta description issue rather than content decay. A human should check the SERP presentation. | Directional — rule-based signal |
| **Weak/low signal** | `monitor` (score < 0.55) | **monitor** — no action now | The model did not observe strong decline signals. These items stay in the queue for the next scoring cycle. No refresh is recommended at this time. | Weak — absence of signal |

**Important:** Every action above requires a human to verify the page context before proceeding. The model identifies statistical patterns; it does not know whether a page is a revenue driver, a legal requirement, or a seasonal dip.

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

**Intended use:** This queue is a *decision-support* tool for the content team. It ranks items by estimated decline probability so the team can prioritize which pages to review first. It does NOT say "refresh this page and traffic will recover." The model flags patterns associated with decline in this dataset — it does not prove causation.

**Who uses it:**
- Content strategists reviewing which pages to refresh each quarter
- SEO leads prioritizing their team's editing backlog
- FlyRank account managers sharing directional insights with clients

**Where it stops being valid:**
- **This is one portfolio, one snapshot.** The model was trained on 30k rows from 32 anonymized clients at a single point in time. Results may not generalize to other industries, geographies, or time periods.
- **The label is noisy.** "Declining" is defined as >20% impression drop in a 30-day window. Some items labeled "declining" may be experiencing normal variance, not a sustained trend.
- **No intervention data.** We have never measured what happens when someone actually follows this queue. The model predicts association, not outcome of action.
- **The model is not real-time.** Scores reflect the trailing 90-day window at export time. A page refreshed yesterday will still score high until the next data pull.

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

### What a human must check before acting

The model scores a page; it does not know the business context. Before refreshing any item from the queue, a person should verify:

1. **Is the page still live?** The data is a snapshot — the page may have been removed, redirected, or noindexed since export.
2. **Does the page still serve a business goal?** A page declining in impressions may still be a critical conversion path. Refreshing it could break conversion funnels.
3. **Is the decline seasonal?** Some content naturally dips in certain months. The model does not account for seasonality.
4. **Are there legal or compliance constraints?** Medical, financial, or regulated content may require expert review before any changes.
5. **What is the actual refresh cost?** A 3,000-word comparison article costs far more to refresh than a 500-word FAQ. The model does not account for cost or effort.

### The no-go list — never automate these

The following actions should **never** be triggered automatically from this queue:

- **Publishing or unpublishing content** — the model identifies risk, not readiness to publish
- **Redirecting URLs** — a wrong redirect is hard to undo and can tank traffic permanently
- **Changing canonical tags or site structure** — these are architectural decisions, not content refreshes
- **Deleting pages** — the model does not measure whether a page has backlinks, internal links, or historical value
- **Sending client-specific recommendations without human review** — each client's business context is different

### Freshness and decay — what the data observed

Across Weeks 4–7, a consistent pattern emerged in the data:

- **Older content declines more often.** In the training data, content with `content_age_days` ≥ 180 was more likely to carry the `is_declining_label` than newer content. The logistic regression assigned a negative coefficient to `content_age_days` (–0.25 in the trained model), meaning older pages were observed to have higher decline probability, all else equal.
- **Stale content with traffic declines more.** The Week 4 baseline flagged items with `days_since_last_update` ≥ 180 and `impressions_90d` ≥ 500 as candidates for refresh. This rule-based signal was among the baseline's two reason codes.
- **Freshness is associated with growth in the research paper.** The FlyRank research paper (Finding #1) observed that growing content is 20% younger (184 vs 230 days) than declining content in the portfolio. This is an observational association, not a controlled experiment.

**What this means for refresh decisions:** These observed associations support reviewing older, stale content first — which is exactly what the `refresh_stale` and `refresh_priority` actions do. A human reviewing these items can assess whether the content is genuinely outdated or whether the decline has another cause (seasonality, competitor changes, algorithm updates).

**What this does NOT prove:** The analysis does not demonstrate that refreshing content causes traffic recovery. We have no intervention data — no A/B test where some pages were refreshed and others were not. The observed association between freshness and performance could be confounded by unmeasured factors (e.g., pages that get refreshed may also receive more internal linking or promotion). Any claim that "refreshing will improve rankings" goes beyond what this observational data supports.

### Cost / Value thinking — a practical prioritization framework

The model provides **evidence strength** (the `model_score`). To turn the queue into a prioritized action plan, a human should also consider two factors the model does **not** contain:

1. **Potential value / opportunity** — How much traffic or revenue could this page recover if it improved? A page targeting a high-volume keyword with many impressions has more upside than a page with minimal search demand.
2. **Estimated intervention effort / cost** — How much work is required to refresh this page? A 500-word FAQ update is far cheaper than a 3,000-word comparison article rewrite. Specialist content (medical, legal) may require expert review, adding cost and time.

**The model does not contain refresh cost or revenue data.** This is a qualitative decision framework, not a measured ROI calculation. Use it to guide human judgment, not to automate prioritization.

| Evidence (model score) | Potential value | Effort / cost | Suggested approach |
|---|---|---|---|
| High (≥ 0.65) | High (high impressions, high-value keyword) | Low / moderate | **Review first** — strong signal + high upside + manageable cost |
| High (≥ 0.65) | High | High (long-form, specialist content) | **Human review before prioritizing** — strong signal but expensive to act on; may need scoping |
| High (≥ 0.65) | Low (low impressions, niche keyword) | Any | **Review, but lower priority** — strong signal but limited upside |
| Moderate (0.55–0.65) | High | Low / moderate | **Review after high-score items** — decent signal with good upside |
| Moderate (0.55–0.65) | Any | High | **Defer or batch** — moderate signal, high cost; wait for more evidence |
| Weak (< 0.55) | Any | Any | **Monitor** — no action recommended at this time |

This framework is a starting point. Each client's context (budget, team capacity, strategic priorities) should adjust the final ordering.

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

The model was trained on a single snapshot. It will go stale. Here are the signals that should trigger a re-evaluation:

| Signal | What to check | Threshold |
|---|---|---|
| **Score drift** | Recompute scores on new data; compare distribution to training-time scores | If the mean or variance of `model_score` shifts by > 10%, investigate |
| **Precision decay** | After a refresh cycle, check whether refreshed items actually improved | If refreshed items show no measurable improvement in 60 days, the model's ranking may be unhelpful |
| **New content types** | If the portfolio adds content types not in the training data (e.g., video pages, tools) | Retrain with the new type represented |
| **Seasonal shifts** | If the portfolio is seasonal (e.g., holiday content), the model's 90-day window may miss cyclical patterns | Consider retraining at the start of each quarter |
| **Client churn** | If major clients leave or new large clients join | The model's learned client patterns may not transfer |

**Practical recommendation:** Retrain the model quarterly on fresh data. Between retraining cycles, use the queue as a *directional guide*, not a fixed plan.

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [8]:
# --- Export the ranked action queue ---
export_cols = ['rank', 'content_id', 'client_id', 'model_score', 'reason_code',
                'action', 'impressions_90d', 'avg_position', 'ctr',
                'days_since_last_update', 'content_age_days', 'content_type']
export_df = queue[export_cols].copy()
export_df.to_csv(OUTPUT_DIR / "action_playbook_queue.csv", index=False)
print(f"Exported {len(export_df):,} rows to work/outputs/action_playbook_queue.csv")
print(f"Columns: {export_df.columns.tolist()}")

Exported 30,000 rows to work/outputs/action_playbook_queue.csv
Columns: ['rank', 'content_id', 'client_id', 'model_score', 'reason_code', 'action', 'impressions_90d', 'avg_position', 'ctr', 'days_since_last_update', 'content_age_days', 'content_type']


In [9]:
# --- Figure 1: Action mix (horizontal bar chart) ---
action_counts = queue['action'].value_counts()
fig, ax = plt.subplots(figsize=(8, 4))
action_counts.plot.barh(ax=ax, color=['#2196F3', '#FF9800', '#4CAF50', '#F44336', '#9C27B0'])
ax.set_xlabel("Number of content items")
ax.set_ylabel("")
ax.set_title("Action Mix — All Content Items")
ax.invert_yaxis()
plt.tight_layout()
fig.savefig(FIG_DIR / "action_mix_w07.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved to {FIG_DIR / 'action_mix_w07.png'}")

Saved to ..\..\work\figures\action_mix_w07.png


C:\Users\amogh\AppData\Local\Temp\ipykernel_23988\1259462635.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [10]:
# --- Figure 2: Model score distribution by action ---
fig, ax = plt.subplots(figsize=(8, 4))
for action in ['refresh_priority', 'refresh', 'refresh_stale', 'refresh_ctr_fix', 'monitor']:
    subset = queue[queue['action'] == action]['model_score']
    if len(subset) > 0:
        ax.hist(subset, bins=30, alpha=0.5, label=f"{action} (n={len(subset):,})", density=True)
ax.set_xlabel("Model decline probability")
ax.set_ylabel("Density")
ax.set_title("Score Distribution by Action")
ax.legend(fontsize=8)
plt.tight_layout()
fig.savefig(FIG_DIR / "score_distribution_by_action.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved to {FIG_DIR / 'score_distribution_by_action.png'}")

Saved to ..\..\work\figures\score_distribution_by_action.png


C:\Users\amogh\AppData\Local\Temp\ipykernel_23988\2190364083.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [11]:
# --- Figure 3: Top feature importance (Logistic Regression) ---
feature_names = X_full.columns.tolist()
coefs = lr_pipeline.named_steps['model'].coef_[0]
importance_df = pd.DataFrame({
    'feature': feature_names,
    'coefficient': coefs,
    'abs_coef': np.abs(coefs),
}).sort_values('abs_coef', ascending=False)

top15 = importance_df.head(15)
fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#F44336' if c > 0 else '#2196F3' for c in top15['coefficient']]
ax.barh(range(len(top15)), top15['coefficient'].values, color=colors)
ax.set_yticks(range(len(top15)))
ax.set_yticklabels(top15['feature'].values, fontsize=8)
ax.set_xlabel("Coefficient (positive = higher decline risk)")
ax.set_title("Top 15 Features — Logistic Regression")
ax.invert_yaxis()
ax.axvline(x=0, color='black', linewidth=0.5)
plt.tight_layout()
fig.savefig(FIG_DIR / "feature_importance_lr.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved to {FIG_DIR / 'feature_importance_lr.png'}")

Saved to ..\..\work\figures\feature_importance_lr.png


C:\Users\amogh\AppData\Local\Temp\ipykernel_23988\826215345.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [12]:
# --- Summary stats for the paper ---
print("=== Action Playbook Summary ===")
print(f"Total items scored: {len(queue):,}")
print(f"Items flagged for review (score >= 0.55): {(queue['model_score'] >= 0.55).sum():,} ({(queue['model_score'] >= 0.55).mean():.1%})")
print(f"Items flagged as high priority (score >= 0.65): {(queue['model_score'] >= 0.65).sum():,} ({(queue['model_score'] >= 0.65).mean():.1%})")
print(f"\nAction breakdown:")
for action, count in queue['action'].value_counts().items():
    print(f"  {action}: {count:,} ({count/len(queue):.1%})")
print(f"\nModel: LogisticRegression (balanced, max_iter=1000)")
print(f"Features: {X_full.shape[1]}")
print(f"Training rows: {X_full.shape[0]:,}")
print(f"Base rate (declining): {y_full.mean():.1%}")
print(f"\nNote: Scores are in-sample (trained and scored on same data).")
print(f"For out-of-sample estimates, see Week 5 holdout (ROC-AUC 0.700) and Week 6 CV (ROC-AUC 0.661).")

=== Action Playbook Summary ===
Total items scored: 30,000
Items flagged for review (score >= 0.55): 13,533 (45.1%)
Items flagged as high priority (score >= 0.65): 7,887 (26.3%)

Action breakdown:
  monitor: 11,400 (38.0%)
  refresh_priority: 7,887 (26.3%)
  refresh: 5,646 (18.8%)
  refresh_ctr_fix: 5,063 (16.9%)
  refresh_stale: 4 (0.0%)

Model: LogisticRegression (balanced, max_iter=1000)
Features: 52
Training rows: 30,000
Base rate (declining): 54.2%

Note: Scores are in-sample (trained and scored on same data).
For out-of-sample estimates, see Week 5 holdout (ROC-AUC 0.700) and Week 6 CV (ROC-AUC 0.661).


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.